# T11 - 机构行为分析

## 项目概述

本项目专注于研究各类机构投资者（银行、保险、基金、券商等）在债券市场的行为特征，包括：
- **持仓偏好分析**：了解各类机构的债券持仓结构和偏好
- **交易模式识别**：把握机构交易节奏和行为特征
- **资金流向跟踪**：识别资金进出方向
- **投资情绪判断**：机构行为反映市场情绪

### 机构投资者类型

| 机构类型 | 投资风格 | 持仓偏好 | 交易特点 |
|---------|---------|---------|---------|
| 商业银行 | 稳健、久期匹配 | 高等级信用债、国债 | 大额、批量交易 |
| 保险公司 | 长期持有、匹配负债 | 超长期信用债、高等级债券 | 持有到期 |
| 公募基金 | 相对收益、动态调整 | 信用利差策略 | 定期调仓 |
| 券商自营 | 追求收益、杠杆交易 | 中低等级信用债、可转债 | 高频交易 |

---

## 1. 环境配置

导入所需依赖包和配置参数。

In [ ]:
# 1.1 导入基础库
import os
import sys
import datetime
import warnings
warnings.filterwarnings('ignore')

# 1.2 导入数据处理库
import pandas as pd
import numpy as np

# 1.3 导入可视化库
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# 1.4 导入数据库连接库
from sqlalchemy import create_engine

# 1.5 显示配置
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 100)

print("环境配置完成！")
print(f"当前时间: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2. 配置参数

加载配置文件和环境变量。

In [ ]:
# 2.1 加载配置文件
from config import Config

# 2.2 初始化配置
config = Config()

# 2.3 显示配置信息
print("=== 配置信息 ===")
print(f"数据目录: {config.data_dir}")
print(f"输出目录: {config.output_dir}")
print(f"目标机构: {config.target_institutions}")
print(f"分析日期分割点: {config.analysis_split_date}")

## 3. 数据获取

### 3.1 数据源说明
- **回购交易数据**：正逆回购利率、金额、余额等
- **现券交易数据**：净买入、买入、卖出交易量等
- **机构类型**：货币市场基金、理财子公司、基金公司等非银机构

In [ ]:
def load_repo_data(file_path):
    """
    加载回购交易数据
    
    Parameters:
    -----------
    file_path : str
        数据文件路径
        
    Returns:
    --------
    pd.DataFrame
        回购交易数据
    """
    try:
        data = pd.read_excel(file_path, parse_dates=['交易日期'])
        print(f"回购数据加载成功，形状: {data.shape}")
        return data
    except Exception as e:
        print(f"回购数据加载失败: {e}")
        return None


def load_bond_data(file_path):
    """
    加载现券交易数据
    
    Parameters:
    -----------
    file_path : str
        数据文件路径
        
    Returns:
    --------
    pd.DataFrame
        现券交易数据
    """
    try:
        data = pd.read_excel(file_path, sheet_name='数据', parse_dates=['交易日期'])
        print(f"现券数据加载成功，形状: {data.shape}")
        return data
    except Exception as e:
        print(f"现券数据加载失败: {e}")
        return None


# 加载数据
repo_file = os.path.join(config.data_dir, config.repo_data_file)
bond_file = os.path.join(config.data_dir, config.bond_data_file)

repo_data = load_repo_data(repo_file)
bond_data = load_bond_data(bond_file)

## 4. 数据处理

### 4.1 数据清洗
- 处理缺失值和非数值数据
- 单位转换（百万 -> 亿）
- 机构类型映射

In [ ]:
# 4.1 机构类型映射表
INSTITUTION_MAPPING = {
    '大型商业银行/政策性银行': 'Large Commercial/Policy Bank',
    '股份制商业银行': 'Joint-stock Commercial Bank',
    '城市商业银行': 'City Commercial Bank',
    '农村金融机构': 'Rural Financial Institution',
    '外资银行': 'Foreign Bank',
    '证券公司': 'Securities Company',
    '保险公司': 'Insurance Company',
    '基金公司及产品': 'Fund Company & Products',
    '货币市场基金': 'Monetary Market Fund',
    '理财子公司及理财类产品': 'Wealth Management Subsidiary & Products',
    '其他产品类': 'Other Products',
    '其他': 'Others',
    '其它': 'Others'
}

# 期限映射表
MATURITY_MAPPING = {
    '≦1Y': '<=1Y',
    '1-3Y': '1-3Y', 
    '3-5Y': '3-5Y',
    '5-7Y': '5-7Y',
    '7-10Y': '7-10Y',
    '10-15Y': '10-15Y',
    '15-20Y': '15-20Y',
    '20-30Y': '20-30Y',
    '>30Y': '>30Y'
}

# 债券类型映射表
BOND_TYPE_MAPPING = {
    '国债-新债': 'Treasury-New',
    '国债-老债': 'Treasury-Old',
    '政策性金融债-新债': 'Policy Bank Bond-New',
    '政策性金融债-老债': 'Policy Bank Bond-Old',
    '中期票据': 'Medium-term Note',
    '短期/超短期融资券': 'Short-term/Ultra-short CP',
    '企业债': 'Corporate Bond',
    '地方政府债': 'Local Government Bond',
    '同业存单': 'NCD',
    '资产支持证券': 'ABS',
    '其他': 'Others'
}

print("映射表定义完成！")

In [ ]:
def clean_repo_data(data):
    """
    清洗回购数据
    
    Parameters:
    -----------
    data : pd.DataFrame
        原始回购数据
        
    Returns:
    --------
    pd.DataFrame
        清洗后的数据
    """
    if data is None:
        return None
    
    df = data.copy()
    
    # 转换金额单位为亿元
    amount_cols = [col for col in df.columns if '百万' in col]
    for col in amount_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col] / 100
        new_col = col.replace('(百万)', '(亿)')
        df = df.rename(columns={col: new_col})
    
    # 处理利率数据
    rate_cols = [col for col in df.columns if '利率' in col]
    for col in rate_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # 添加英文机构类型
    df['Institution_Type_EN'] = df['机构类型'].map(INSTITUTION_MAPPING)
    
    return df


def clean_bond_data(data):
    """
    清洗现券数据
    
    Parameters:
    -----------
    data : pd.DataFrame
        原始现券数据
        
    Returns:
    --------
    pd.DataFrame
        清洗后的数据
    """
    if data is None:
        return None
    
    df = data.copy()
    
    # 处理数值列
    numeric_cols = ['净买入交易量（亿元）', '买入交易量（亿元）', '卖出交易量（亿元）']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = df[col].replace('--', np.nan)
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # 添加英文标识列
    df['Institution_Type_EN'] = df['机构类型'].map(INSTITUTION_MAPPING)
    if '期限' in df.columns:
        df['Maturity_EN'] = df['期限'].map(MATURITY_MAPPING)
    if '债券类型' in df.columns:
        df['Bond_Type_EN'] = df['债券类型'].map(BOND_TYPE_MAPPING)
    
    return df


# 执行数据清洗
repo_data_clean = clean_repo_data(repo_data)
bond_data_clean = clean_bond_data(bond_data)

print("=== 数据清洗完成 ===")
if repo_data_clean is not None:
    print(f"回购数据: {repo_data_clean.shape}")
if bond_data_clean is not None:
    print(f"现券数据: {bond_data_clean.shape}")

In [ ]:
# 4.2 添加期间标识
def add_period_identifier(data, split_date):
    """
    添加期间标识（用于前后对比分析）
    
    Parameters:
    -----------
    data : pd.DataFrame
        数据
    split_date : str
        分割日期
        
    Returns:
    --------
    pd.DataFrame
        带期间标识的数据
    """
    if data is None:
        return None
    
    df = data.copy()
    df['Period'] = df['交易日期'].apply(
        lambda x: 'Recent' if x >= pd.to_datetime(split_date) else 'Earlier'
    )
    return df


repo_data_clean = add_period_identifier(repo_data_clean, config.analysis_split_date)
bond_data_clean = add_period_identifier(bond_data_clean, config.analysis_split_date)

print("期间标识添加完成！")
if repo_data_clean is not None:
    print(f"近期数据: {(repo_data_clean['Period'] == 'Recent').sum()} 条")
    print(f"前期数据: {(repo_data_clean['Period'] == 'Earlier').sum()} 条")

## 5. 核心逻辑

### 5.1 逆回购利率趋势分析

In [ ]:
def analyze_repo_rate_trend(data, target_institutions):
    """
    分析逆回购利率变化趋势
    
    Parameters:
    -----------
    data : pd.DataFrame
        回购数据
    target_institutions : list
        目标机构列表
        
    Returns:
    --------
    pd.DataFrame
        分析结果
    """
    if data is None:
        return None
    
    # 筛选目标机构
    target_data = data[
        data['Institution_Type_EN'].isin(target_institutions)
    ].copy()
    
    # 按日期和机构类型聚合
    daily_avg = target_data.groupby(['交易日期', 'Institution_Type_EN']).agg({
        '逆回购加权利率(%)': 'mean',
        '逆回购金额(亿)': 'sum',
        '正回购加权利率(%)': 'mean',
        '正回购金额(亿)': 'sum'
    }).reset_index()
    
    return daily_avg


# 执行逆回购利率分析
repo_analysis = analyze_repo_rate_trend(repo_data_clean, config.target_institutions)

if repo_analysis is not None:
    print("=== 逆回购利率分析结果 ===")
    print(f"分析数据形状: {repo_analysis.shape}")
    print(repo_analysis.head(10))

In [ ]:
# 5.2 计算利率变化统计
def calculate_rate_statistics(data, split_date):
    """
    计算利率变化统计
    
    Parameters:
    -----------
    data : pd.DataFrame
        分析数据
    split_date : str
        分割日期
        
    Returns:
    --------
    dict
        统计结果
    """
    if data is None:
        return None
    
    recent = data[data['交易日期'] >= pd.to_datetime(split_date)]
    early = data[data['交易日期'] < pd.to_datetime(split_date)]
    
    stats = {
        'recent_avg_rate': recent['逆回购加权利率(%)'].mean(),
        'early_avg_rate': early['逆回购加权利率(%)'].mean(),
        'recent_total_amount': recent['逆回购金额(亿)'].sum(),
        'early_total_amount': early['逆回购金额(亿)'].sum()
    }
    
    stats['rate_change'] = stats['recent_avg_rate'] - stats['early_avg_rate']
    stats['amount_change_pct'] = (
        (stats['recent_total_amount'] - stats['early_total_amount']) / 
        (stats['early_total_amount'] + 1e-10) * 100
    )
    
    return stats


rate_stats = calculate_rate_statistics(repo_analysis, config.analysis_split_date)

if rate_stats:
    print("=== 利率变化统计 ===")
    print(f"近期平均利率: {rate_stats['recent_avg_rate']:.4f}%")
    print(f"前期平均利率: {rate_stats['early_avg_rate']:.4f}%")
    print(f"利率变化: {rate_stats['rate_change']:.4f} 个基点")
    print(f"金额变化: {rate_stats['amount_change_pct']:.2f}%")

### 5.2 现券交易分析

In [ ]:
def analyze_bond_trading(data, target_institutions):
    """
    分析现券交易变化
    
    Parameters:
    -----------
    data : pd.DataFrame
        现券数据
    target_institutions : list
        目标机构列表
        
    Returns:
    --------
    pd.DataFrame
        分析结果
    """
    if data is None:
        return None
    
    # 筛选目标机构
    target_data = data[
        data['Institution_Type_EN'].isin(target_institutions)
    ].copy()
    
    # 按日期和机构类型聚合
    daily_bond = target_data.groupby(['交易日期', 'Institution_Type_EN']).agg({
        '净买入交易量（亿元）': 'sum',
        '买入交易量（亿元）': 'sum',
        '卖出交易量（亿元）': 'sum'
    }).reset_index()
    
    return daily_bond


# 执行现券交易分析
bond_analysis = analyze_bond_trading(bond_data_clean, config.target_institutions)

if bond_analysis is not None:
    print("=== 现券交易分析结果 ===")
    print(f"分析数据形状: {bond_analysis.shape}")
    print(bond_analysis.head(10))

In [ ]:
# 5.3 计算现券交易统计
def calculate_bond_statistics(data, split_date):
    """
    计算现券交易统计
    
    Parameters:
    -----------
    data : pd.DataFrame
        分析数据
    split_date : str
        分割日期
        
    Returns:
    --------
    dict
        统计结果
    """
    if data is None:
        return None
    
    recent = data[data['交易日期'] >= pd.to_datetime(split_date)]
    early = data[data['交易日期'] < pd.to_datetime(split_date)]
    
    stats = {
        'recent_net_purchase': recent['净买入交易量（亿元）'].sum(),
        'early_net_purchase': early['净买入交易量（亿元）'].sum(),
        'recent_buy': recent['买入交易量（亿元）'].sum(),
        'early_buy': early['买入交易量（亿元）'].sum(),
        'recent_sell': recent['卖出交易量（亿元）'].sum(),
        'early_sell': early['卖出交易量（亿元）'].sum()
    }
    
    stats['net_change'] = stats['recent_net_purchase'] - stats['early_net_purchase']
    
    return stats


bond_stats = calculate_bond_statistics(bond_analysis, config.analysis_split_date)

if bond_stats:
    print("=== 现券交易统计 ===")
    print(f"近期净买入: {bond_stats['recent_net_purchase']:.2f} 亿元")
    print(f"前期净买入: {bond_stats['early_net_purchase']:.2f} 亿元")
    print(f"净买入变化: {bond_stats['net_change']:.2f} 亿元")

### 5.3 市场集中度分析

In [ ]:
def calculate_market_concentration(data, target_institutions, split_date):
    """
    计算市场集中度变化
    
    Parameters:
    -----------
    data : pd.DataFrame
        回购数据
    target_institutions : list
        目标机构列表
    split_date : str
        分割日期
        
    Returns:
    --------
    dict
        集中度分析结果
    """
    if data is None:
        return None
    
    repo_before = data[data['交易日期'] < pd.to_datetime(split_date)]
    repo_after = data[data['交易日期'] >= pd.to_datetime(split_date)]
    
    def calc_concentration(df):
        total_amount = df.groupby('Institution_Type_EN')['逆回购金额(亿)'].sum()
        target_amount = total_amount[total_amount.index.isin(target_institutions)].sum()
        return target_amount / total_amount.sum() if total_amount.sum() > 0 else 0
    
    before_conc = calc_concentration(repo_before)
    after_conc = calc_concentration(repo_after)
    
    return {
        'before_period': before_conc,
        'after_period': after_conc,
        'change': after_conc - before_conc
    }


# 计算市场集中度
concentration = calculate_market_concentration(
    repo_data_clean, config.target_institutions, config.analysis_split_date
)

if concentration:
    print("=== 市场集中度分析 ===")
    print(f"前期集中度: {concentration['before_period']:.4f}")
    print(f"近期集中度: {concentration['after_period']:.4f}")
    print(f"集中度变化: {concentration['change']:.4f}")

### 5.4 债券市场热点分析

In [ ]:
def analyze_market_hotspots(data):
    """
    分析市场净买入热点
    
    Parameters:
    -----------
    data : pd.DataFrame
        现券数据
        
    Returns:
    --------
    dict
        热点分析结果
    """
    if data is None:
        return None
    
    results = {}
    
    # 按期限分析
    if 'Maturity_EN' in data.columns:
        maturity_analysis = data.groupby(['交易日期', 'Maturity_EN']).agg({
            '买入交易量（亿元）': 'sum',
            '卖出交易量（亿元）': 'sum',
            '净买入交易量（亿元）': 'sum'
        }).reset_index()
        results['maturity_analysis'] = maturity_analysis
    
    # 按债券类型分析
    if 'Bond_Type_EN' in data.columns:
        bond_type_analysis = data.groupby(['交易日期', 'Bond_Type_EN']).agg({
            '买入交易量（亿元）': 'sum',
            '卖出交易量（亿元）': 'sum',
            '净买入交易量（亿元）': 'sum'
        }).reset_index()
        results['bond_type_analysis'] = bond_type_analysis
    
    # 计算趋势变化
    if 'Maturity_EN' in data.columns:
        maturity_trend = data.groupby(['Maturity_EN', 'Period']).agg({
            '净买入交易量（亿元）': 'sum'
        }).reset_index()
        maturity_pivot = maturity_trend.pivot(
            index='Maturity_EN', columns='Period', values='净买入交易量（亿元）'
        ).fillna(0)
        if 'Earlier' in maturity_pivot.columns and 'Recent' in maturity_pivot.columns:
            maturity_pivot['Change'] = maturity_pivot['Recent'] - maturity_pivot['Earlier']
        results['maturity_trend'] = maturity_pivot
    
    if 'Bond_Type_EN' in data.columns:
        bond_type_trend = data.groupby(['Bond_Type_EN', 'Period']).agg({
            '净买入交易量（亿元）': 'sum'
        }).reset_index()
        bond_type_pivot = bond_type_trend.pivot(
            index='Bond_Type_EN', columns='Period', values='净买入交易量（亿元）'
        ).fillna(0)
        if 'Earlier' in bond_type_pivot.columns and 'Recent' in bond_type_pivot.columns:
            bond_type_pivot['Change'] = bond_type_pivot['Recent'] - bond_type_pivot['Earlier']
        results['bond_type_trend'] = bond_type_pivot
    
    return results


# 执行热点分析
hotspot_analysis = analyze_market_hotspots(bond_data_clean)

if hotspot_analysis:
    print("=== 债券市场热点分析 ===")
    if 'maturity_trend' in hotspot_analysis:
        print("\n期限净买入变化:")
        print(hotspot_analysis['maturity_trend'])

### 5.5 同业存单分析

In [ ]:
def analyze_certificate_deposit(data):
    """
    分析同业存单
    
    Parameters:
    -----------
    data : pd.DataFrame
        现券数据
        
    Returns:
    --------
    dict
        同业存单分析结果
    """
    if data is None or '债券类型' not in data.columns:
        return None
    
    # 筛选同业存单数据
    cd_data = data[data['债券类型'] == '同业存单'].copy()
    
    if cd_data.empty:
        return None
    
    # 按机构和日期聚合
    daily_cd = cd_data.groupby(['交易日期', 'Institution_Type_EN', '机构类型']).agg({
        '净买入交易量（亿元）': 'sum',
        '买入交易量（亿元）': 'sum',
        '卖出交易量（亿元）': 'sum'
    }).reset_index()
    
    # 获取机构统计
    stats = daily_cd.groupby(['Institution_Type_EN', '机构类型']).agg({
        '净买入交易量（亿元）': ['sum', 'mean', 'std'],
        '买入交易量（亿元）': ['sum', 'mean'],
        '卖出交易量（亿元）': ['sum', 'mean']
    }).round(2)
    
    stats.columns = ['Net_Total', 'Net_Mean', 'Net_Std', 'Buy_Total', 'Buy_Mean', 'Sell_Total', 'Sell_Mean']
    stats = stats.reset_index().sort_values('Net_Total', key=abs, ascending=False)
    
    return {
        'daily_cd': daily_cd,
        'stats': stats
    }


# 执行同业存单分析
cd_analysis = analyze_certificate_deposit(bond_data_clean)

if cd_analysis:
    print("=== 同业存单分析 ===")
    print(f"分析数据形状: {cd_analysis['daily_cd'].shape}")
    print("\n机构统计:")
    print(cd_analysis['stats'].head(10))

## 6. 执行与测试

### 6.1 主流程执行

In [ ]:
def run_main_analysis():
    """
    执行主分析流程
    
    Returns:
    --------
    dict
        分析结果
    """
    print("=== 金融市场机构行为分析 ===")
    print(f"开始时间: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    results = {}
    
    # 1. 逆回购利率分析
    print("\n1. 逆回购利率分析...")
    results['repo_analysis'] = analyze_repo_rate_trend(repo_data_clean, config.target_institutions)
    results['rate_stats'] = calculate_rate_statistics(repo_analysis, config.analysis_split_date)
    
    # 2. 现券交易分析
    print("2. 现券交易分析...")
    results['bond_analysis'] = analyze_bond_trading(bond_data_clean, config.target_institutions)
    results['bond_stats'] = calculate_bond_statistics(bond_analysis, config.analysis_split_date)
    
    # 3. 市场集中度分析
    print("3. 市场集中度分析...")
    results['concentration'] = calculate_market_concentration(
        repo_data_clean, config.target_institutions, config.analysis_split_date
    )
    
    # 4. 热点分析
    print("4. 债券市场热点分析...")
    results['hotspot'] = analyze_market_hotspots(bond_data_clean)
    
    # 5. 同业存单分析
    print("5. 同业存单分析...")
    results['cd_analysis'] = analyze_certificate_deposit(bond_data_clean)
    
    print(f"\n=== 分析完成 ===")
    print(f"结束时间: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    return results


# 执行主分析
analysis_results = run_main_analysis()

### 6.2 结果验证

In [ ]:
def validate_results(results):
    """
    验证分析结果
    
    Parameters:
    -----------
    results : dict
        分析结果
        
    Returns:
    --------
    bool
        验证结果
    """
    print("=== 结果验证 ===")
    
    validations = []
    
    # 验证逆回购分析
    if results.get('repo_analysis') is not None:
        print("[PASS] 逆回购利率分析完成")
        validations.append(True)
    else:
        print("[FAIL] 逆回购利率分析未完成")
        validations.append(False)
    
    # 验证现券分析
    if results.get('bond_analysis') is not None:
        print("[PASS] 现券交易分析完成")
        validations.append(True)
    else:
        print("[FAIL] 现券交易分析未完成")
        validations.append(False)
    
    # 验证市场集中度
    if results.get('concentration') is not None:
        print("[PASS] 市场集中度分析完成")
        validations.append(True)
    else:
        print("[FAIL] 市场集中度分析未完成")
        validations.append(False)
    
    # 验证热点分析
    if results.get('hotspot') is not None:
        print("[PASS] 债券市场热点分析完成")
        validations.append(True)
    else:
        print("[FAIL] 债券市场热点分析未完成")
        validations.append(False)
    
    # 验证同业存单分析
    if results.get('cd_analysis') is not None:
        print("[PASS] 同业存单分析完成")
        validations.append(True)
    else:
        print("[WARN] 同业存单分析未完成（可能无数据）")
        validations.append(True)  # 不作为失败条件
    
    return all(validations)


# 执行验证
validation_passed = validate_results(analysis_results)
print(f"\n验证结果: {'通过' if validation_passed else '失败'}")

### 6.3 生成分析摘要

In [ ]:
def generate_summary(results):
    """
    生成分析摘要
    
    Parameters:
    -----------
    results : dict
        分析结果
        
    Returns:
    --------
    str
        分析摘要
    """
    summary = """# 机构行为分析摘要

## 核心发现

"""
    
    # 利率分析
    if results.get('rate_stats'):
        rate = results['rate_stats']
        summary += f"""### 1. 逆回购利率分析
- 近期平均利率: {rate['recent_avg_rate']:.4f}%
- 前期平均利率: {rate['early_avg_rate']:.4f}%
- 利率变化: {rate['rate_change']:.4f} 个基点
- 金额变化: {rate['amount_change_pct']:.2f}%

"""
    
    # 现券分析
    if results.get('bond_stats'):
        bond = results['bond_stats']
        summary += f"""### 2. 现券交易分析
- 近期净买入: {bond['recent_net_purchase']:.2f} 亿元
- 前期净买入: {bond['early_net_purchase']:.2f} 亿元
- 净买入变化: {bond['net_change']:.2f} 亿元

"""
    
    # 集中度分析
    if results.get('concentration'):
        conc = results['concentration']
        summary += f"""### 3. 市场集中度分析
- 前期集中度: {conc['before_period']:.4f}
- 近期集中度: {conc['after_period']:.4f}
- 集中度变化: {conc['change']:.4f}

"""
    
    summary += f"""---
生成时间: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
"""
    
    return summary


# 生成并显示摘要
summary = generate_summary(analysis_results)
print(summary)

## 7. 工具函数

可复用的辅助函数集合。

In [ ]:
# 7.1 机构类型翻译
INSTITUTION_TRANSLATION = {
    'Large Commercial/Policy Bank': '大型商业/政策性银行',
    'Joint-stock Commercial Bank': '股份制商业银行',
    'City Commercial Bank': '城市商业银行',
    'Rural Financial Institution': '农村金融机构',
    'Foreign Bank': '外资银行',
    'Securities Company': '证券公司',
    'Insurance Company': '保险公司',
    'Fund Company & Products': '基金公司及产品',
    'Monetary Market Fund': '货币市场基金',
    'Wealth Management Subsidiary & Products': '理财子公司及理财类产品',
    'Other Products': '其他产品类',
    'Others': '其他'
}


def translate_institution(en_name):
    """
    将英文机构类型翻译为中文
    
    Parameters:
    -----------
    en_name : str
        英文机构类型
        
    Returns:
    --------
    str
        中文机构类型
    """
    return INSTITUTION_TRANSLATION.get(en_name, en_name)


def get_top_institutions(data, column, n=10):
    """
    获取按某列排序的前N个机构
    
    Parameters:
    -----------
    data : pd.DataFrame
        数据
    column : str
        排序列
    n : int
        返回数量
        
    Returns:
    --------
    pd.DataFrame
        排序后的数据
    """
    return data.nlargest(n, column)


def calculate_change_percentage(recent, earlier):
    """
    计算变化百分比
    
    Parameters:
    -----------
    recent : float
        近期值
    earlier : float
        前期值
        
    Returns:
    --------
    float
        变化百分比
    """
    if earlier == 0:
        return 0
    return ((recent - earlier) / abs(earlier)) * 100


def save_analysis_results(results, output_dir):
    """
    保存分析结果
    
    Parameters:
    -----------
    results : dict
        分析结果
    output_dir : str
        输出目录
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    for name, data in results.items():
        if isinstance(data, pd.DataFrame):
            file_path = os.path.join(output_dir, f"{name}.csv")
            data.to_csv(file_path, index=False, encoding='utf-8-sig')
            print(f"已保存: {file_path}")


print("工具函数定义完成！")

## 8. 总结

本Notebook完成了以下功能：

1. **数据获取与处理**
   - 加载回购交易和现券交易数据
   - 数据清洗和转换
   - 机构类型映射

2. **核心分析模块**
   - 逆回购利率趋势分析
   - 现券交易行为分析
   - 市场集中度计算
   - 债券市场热点分析
   - 同业存单专项分析

3. **结果输出**
   - 分析结果验证
   - 分析摘要生成
   - 结果数据导出

---

**生成时间**: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

**任务ID**: T11

**任务名称**: 机构行为